In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.integrate import solve_ivp
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error

# -------------------------
# Model components
# -------------------------
class FourierFeatures(nn.Module):
    def __init__(self, in_features=1, out_features=64, scale=10.0, device=None, dtype=torch.float32):
        super().__init__()
        B = torch.randn((out_features, in_features), device=device, dtype=dtype) * scale
        self.register_buffer('B', B)
        self.register_buffer('two_pi', torch.tensor(2.0 * np.pi, device=device, dtype=dtype))

    def forward(self, x):
        x_proj = self.two_pi * (x @ self.B.T)
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)


class ResidualBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.fc1 = nn.Linear(width, width)
        self.fc2 = nn.Linear(width, width)

    def forward(self, x):
        return x + self.fc2(torch.sin(self.fc1(x)))


class F_PINN(nn.Module):
    def __init__(self, N_fields, embed_dim=64, hidden=128, blocks=3, fourier_scale=10.0, device=None, dtype=torch.float32):
        super().__init__()
        self.N_fields = N_fields
        self.fourier = FourierFeatures(1, embed_dim, scale=fourier_scale, device=device, dtype=dtype)
        layers = [nn.Linear(2 * embed_dim, hidden)]
        for _ in range(blocks):
            layers.append(ResidualBlock(hidden))
        self.net = nn.Sequential(*layers)
        self.output = nn.Linear(hidden, 1 + N_fields)
        self.softplus = nn.Softplus()

    def forward(self, t):
        x = self.fourier(t)
        x = self.net(x)
        x = self.output(x)
        a = self.softplus(x[:, 0:1]).clamp(min=1e-6)
        phi = x[:, 1:]
        return a, phi

# -------------------------
# PINN solver
# -------------------------
class PINNSolver:
    def __init__(self, N_fields=1, m_vec=[10],
                 rho_m0=0.81, rho_r0=0.00027138, rho_l0=2.19,
                 a0=1e-8, phi0=None, phi_dot0=None,
                 t_span=(0.0, 1.0), t_eval=None, folder_name='results',
                 device=None, dtype=torch.float32):

        self.device = device or (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
        self.dtype = dtype
        print("Using device:", self.device)

        self.N_fields = int(N_fields)
        self.m_vec = np.array(m_vec if m_vec is not None else [20.0] * self.N_fields, dtype=np.float32)
        self.rho_m0 = rho_m0
        self.rho_r0 = rho_r0
        self.rho_l0 = rho_l0

        self.a0 = float(a0)
        self.phi0 = np.array(phi0 if phi0 is not None else [1.0] * self.N_fields, dtype=np.float32)
        self.phi_dot0 = np.array(phi_dot0 if phi_dot0 is not None else [0.0] * self.N_fields, dtype=np.float32)
        self.y0 = np.concatenate([[self.a0], self.phi0, self.phi_dot0]).astype(np.float32)

        self.t_span = t_span
        if t_eval is None:
            self.t_eval = np.logspace(np.log10(self.a0), np.log10(float(self.t_span[1]) + 1e-12), 1000).astype(np.float32)
        else:
            self.t_eval = np.array(t_eval, dtype=np.float32)

        os.makedirs(folder_name, exist_ok=True)
        self.folder_name = folder_name

        # model
        self.model = F_PINN(self.N_fields, device=self.device, dtype=self.dtype).to(self.device, dtype=self.dtype)

        # torch constants
        self.m_vec_torch = torch.tensor(self.m_vec, dtype=self.dtype, device=self.device).reshape(1, -1)
        self.phi0_torch = torch.tensor(self.phi0.reshape(1, -1), dtype=self.dtype, device=self.device)
        self.a0_torch = torch.tensor([[self.a0]], dtype=self.dtype, device=self.device)
        self.t0_torch = torch.tensor([[0.0]], dtype=self.dtype, device=self.device)
        self.eps = torch.tensor(1e-12, dtype=self.dtype, device=self.device)

    # ---------- ODE reference ----------
    def ode_system(self, t, y):
        N = self.N_fields
        a = y[0]
        phi = y[1:N + 1]
        phi_dot = y[N + 1:2 * N + 1]
        kinetic = 0.5 * np.sum((phi_dot * a) ** 2)
        potential = 0.5 * np.sum((self.m_vec ** 2) * (phi * a) ** 2)
        H = np.sqrt((1.0 / 3.0) * (kinetic + potential + self.rho_m0 / a + self.rho_r0 / a ** 2 + self.rho_l0 * a ** 2))
        a_dot = H
        phi_ddot = - np.sqrt(3.0) * np.sqrt(
            0.5 * np.sum(phi_dot ** 2) +
            0.5 * np.sum((self.m_vec ** 2) * phi ** 2) +
            self.rho_m0 / a ** 3 + self.rho_r0 / a ** 4 + self.rho_l0
        ) * phi_dot - (self.m_vec ** 2) * phi

        dydt = np.zeros_like(y)
        dydt[0] = a_dot
        dydt[1:N + 1] = phi_dot
        dydt[N + 1:2 * N + 1] = phi_ddot
        return dydt

    def solve_ode(self):
        print("Solving ODE reference solution...")
        t0 = time.time()
        sol = solve_ivp(self.ode_system, self.t_span, self.y0, t_eval=self.t_eval, method='RK45', rtol=1e-6, atol=1e-9)
        print(f"ODE solve time: {time.time() - t0:.2f} s")
        self.a_sol = sol.y[0, :].astype(np.float32)
        self.phi_sol = sol.y[1:1 + self.N_fields, :].astype(np.float32)

    # ---------- Physics loss ----------
    def physics_loss(self, t_batch):
        a, phi = self.model(t_batch)
        a_t = torch.autograd.grad(a, t_batch, torch.ones_like(a), create_graph=True, allow_unused=True)[0]
        phi_t = torch.autograd.grad(phi, t_batch, torch.ones_like(phi), create_graph=True, allow_unused=True)[0]
        phi_tt = torch.autograd.grad(phi_t, t_batch, torch.ones_like(phi_t), create_graph=True, allow_unused=True)[0]

        # fallback for unused
        a_t = torch.zeros_like(a) if a_t is None else a_t
        phi_t = torch.zeros_like(phi) if phi_t is None else phi_t
        phi_tt = torch.zeros_like(phi) if phi_tt is None else phi_tt

        kinetic = 0.5 * torch.sum((phi_t ** 2) * (a ** 2), dim=1, keepdim=True)
        potential = 0.5 * torch.sum((self.m_vec_torch ** 2) * (phi ** 2) * (a ** 2), dim=1, keepdim=True)
        Friedmann = a_t - torch.sqrt((1.0 / 3.0) * (kinetic + potential + self.rho_m0 / a + self.rho_r0 / (a ** 2) + self.rho_l0 * (a ** 2)) + self.eps)
        sqsumrho = torch.sqrt(torch.tensor(3.0, device=self.device, dtype=self.dtype)) * torch.sqrt(
            0.5 * torch.sum(phi_t ** 2, dim=1, keepdim=True) +
            0.5 * torch.sum((self.m_vec_torch ** 2) * (phi ** 2), dim=1, keepdim=True) +
            self.rho_m0 / (a ** 3) + self.rho_r0 / (a ** 4) + self.rho_l0 + self.eps
        )
        KG = phi_tt + sqsumrho * phi_t + (self.m_vec_torch ** 2) * phi
        return torch.mean(Friedmann ** 2) + torch.mean(KG ** 2)

    def initial_loss(self):
        a_pred0, phi_pred0 = self.model(self.t0_torch)
        return 4.0 * torch.mean((a_pred0 - self.a0_torch) ** 2) + 30.0 * torch.mean((phi_pred0 - self.phi0_torch) ** 2)

    # ---------- Adam training ----------
    def train(self, max_epochs_adam=1200, physics_weight=10.0, ic_weight=300.0,
              N_f=2000, collocation_batch=512, print_every=200, lr=1e-4):
        optimizer = optim.Adam(self.model.parameters(), lr=lr)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2000, gamma=0.4)
        scaler = torch.cuda.amp.GradScaler(enabled=(self.device.type == 'cuda'))
        self.loss_history = []

        t_min = 0.0
        t_max = 1.0

        print("Training PINN (Adam)...")
        t_start = time.time()

        for epoch in range(max_epochs_adam):
            idx = np.random.randint(0, N_f, collocation_batch)
            t_f = torch.linspace(t_min, t_max, N_f, device=self.device, dtype=self.dtype).reshape(-1, 1)
            t_batch = t_f[idx].clone().requires_grad_()

            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(self.device.type == 'cuda')):
                loss_physics_val = self.physics_loss(t_batch)
                loss_ic_val = self.initial_loss()
                loss = physics_weight * loss_physics_val + ic_weight * loss_ic_val

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            self.loss_history.append(loss.item())

            if epoch % print_every == 0 or epoch == max_epochs_adam - 1:
                percent = 100 * epoch / max_epochs_adam
                print(f"Epoch {epoch:5d}/{max_epochs_adam} ({percent:5.1f}%) | Total Loss: {loss.item():.3e}")

        print(f"Adam training finished in {time.time() - t_start:.2f} s")

    # ---------- LBFGS optimization ----------
    def optimize_lbfgs(self, physics_weight=10.0, ic_weight=300.0, N_f=1000):
        print("Starting LBFGS optimization...")
        optimizer_lbfgs = optim.LBFGS(
            self.model.parameters(),
            lr=1.0,
            max_iter=5000,
            tolerance_grad=1e-9,
            tolerance_change=1e-10,
            history_size=100,
            line_search_fn='strong_wolfe'
        )

        t0 = time.time()
        t_f = torch.linspace(0.0, 1.0, N_f, device=self.device, dtype=self.dtype).reshape(-1, 1).requires_grad_()

        def closure():
            optimizer_lbfgs.zero_grad()
            loss = physics_weight * self.physics_loss(t_f) + ic_weight * self.initial_loss()
            loss.backward()
            return loss

        optimizer_lbfgs.step(closure)
        print(f"LBFGS optimization time: {time.time() - t0:.2f} s")


In [ ]:
if __name__ == "__main__":
    solver = PINNSolver(N_fields=1)  # example with 2 fields

    t_start_total = time.time()

    # ODE solve
    t0 = time.time()
    solver.solve_ode()
    print(f"Time for ODE solve: {time.time() - t0:.2f} s")

    # Adam training
    t0 = time.time()
    solver.train(max_epochs_adam=7000, collocation_batch=512)
    print(f"Time for Adam training: {time.time() - t0:.2f} s")

    # LBFGS optimization
    t0 = time.time()
    print(f"Time for LBFGS: __________")
    solver.optimize_lbfgs(N_f=400)
    print(f"Time for LBFGS: {time.time() - t0:.2f} s")

    # Evaluation
    t0 = time.time()
    t_plot = torch.tensor(solver.t_eval, device=solver.device, dtype=solver.dtype).reshape(-1, 1).requires_grad_(True)
    solver.a_pred, solver.phi_pred = solver.model(t_plot)
    solver.a_pred = solver.a_pred.detach().cpu().numpy().flatten()
    solver.phi_pred = solver.phi_pred.detach().cpu().numpy().T
    print(f"Time for evaluation: {time.time() - t0:.2f} s")

    print(f"Total pipeline time: {time.time() - t_start_total:.2f} s")


In [ ]:
# -------------------------
# Evaluation & plotting
# -------------------------
def evaluate_and_plot(solver):
    t0 = time.time()
    # Prepare collocation points
    t_plot = torch.tensor(solver.t_eval, device=solver.device, dtype=solver.dtype).reshape(-1, 1).requires_grad_(True)
    a_pred, phi_pred = solver.model(t_plot)
    solver.a_pred = a_pred.detach().cpu().numpy().flatten()
    solver.phi_pred = phi_pred.detach().cpu().numpy().T
    print(f"Evaluation time: {time.time() - t0:.2f} s")

    # ---------- Plot ODE vs PINN ----------
    plt.figure(dpi=120)
    plt.plot(solver.t_eval, solver.a_sol, 'k--', label='ODE a(t)')
    plt.plot(solver.t_eval, solver.a_pred, 'r-', label='PINN a(t)')
    for i in range(min(solver.N_fields, 5)):
        plt.plot(solver.t_eval, solver.phi_sol[i, :], 'k--', alpha=0.5)
        plt.plot(solver.t_eval, solver.phi_pred[i, :], 'r-', alpha=0.5)
    plt.xscale('log')
    plt.xlabel('t')
    plt.ylabel('Value')
    plt.title('Comparison: ODE vs PINN')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(solver.folder_name, 'comparison.png'))
    plt.show()

    # ---------- Plot loss curves ----------
    if hasattr(solver, 'loss_history') and solver.loss_history:
        plt.figure(dpi=120)
        plt.plot(solver.loss_history, label='Total Loss')
        plt.yscale('log')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training Loss')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(solver.folder_name, 'loss_curves.png'))
        plt.show()
